Document Q&A AI System

Project objective

The goal of this project is to build an AI application that can answer questions based on the contents of an uploaded PDF document.

How it will work

Upload a PDF document.

Extract text from the PDF.

Divide the text into smaller chunks.

Convert the chunks into embeddings.

Find the most relevant chunks for a user's question.

Use a language model to generate an answer based on the retrieved information.

Tools and technologies

Python

Google Colab

PyMuPDF for PDF text extraction

Sentence Transformers for embeddings

A vector search tool and a language model, which we will add later.

Current scope: Process one PDF at a time. A user can replace it with another PDF.

Install PyMuPDF

PyMuPDF is a Python library that helps us open PDF files, read their pages, and extract text. We need it before processing our brochure.

In [3]:
!pip install pymupdf

Extract text from the PDF

In this step, we open the college brochure uploaded to Google Colab and extract readable text from every page. The extracted text will be the input for the next steps.

Text chunking

Chunking divides a long document into smaller pieces. This helps our AI search for relevant information without treating the entire document as one large block.

For this first version, each chunk will contain up to 500 characters. This is a simple character-based method; later we can improve it to preserve sentence boundaries and context.

In [4]:
import pymupdf

pdf_path = "/content/Sri Indu College of Engineering and Technology Broucher.pdf"

doc = pymupdf.open(pdf_path)

full_text = ""

for page in doc:
    full_text += page.get_text()

doc.close()

chunk_size = 500

chunks = [
    full_text[i:i + chunk_size]
    for i in range(0, len(full_text), chunk_size)
]

print("Total characters:", len(full_text))
print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

Total characters: 2050
Total chunks: 5

--- Chunk 1 ---
Sri Indu College of Engineering and Technology 
COLLEGE PRINCIPAL 
Dr.G. Suresh, a zealous teacher and an administrator, has a 19 years of experience in teaching, 
research, and institutional development. He has published two patents, supervisor of two 
Universities, authored a textbook, and published more than 35 papers in refereed international 
journals and conferences. As the Principal of Sri Indu College of Engineering and Technology 
(SICET), he envisions the importance of engineering educ

--- Chunk 2 ---
ation wherein the students learn, apply 
and develop innovative solutions /products that solve the problems of societal needy. 
Admission Criteria for UG Programs 
The candidates who want to seek admission in SICET have to qualify in the EAMCET (Engineering, 
Agricultural and Medical Common Entrance Test) examination conducted by Govt. of TS. To be 
eligible for appearing in the EAMCET examination, the student should have 

Prepare for embeddings
An embedding is a numerical representation of text. An embedding model helps us find related text even when the question uses different words from the PDF.
We will use Sentence Transformers and its pretrained all-MiniLM-L6-v2 model. It can run in Colab without requiring an API key.

In [5]:
!pip install -q sentence-transformers


# Generate embeddings for your PDF chunks
Convert chunks into embeddings
We will convert each chunk extracted from the brochure into a numerical vector. These vectors will later help us retrieve the most relevant chunks for a user's question.
Each chunk gets its own embedding. This step does not generate answers yet; it prepares the document for semantic search.

In [6]:
from sentence_transformers import SentenceTransformer

# Load the pretrained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [7]:
chunk_embeddings = model.encode(
    chunks,
    convert_to_tensor=True
)

print("Retrieval embeddings ready!")
print("Embedding shape:", chunk_embeddings.shape)

Retrieval embeddings ready!
Embedding shape: torch.Size([5, 384])


In [8]:
import torch
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device=0,
    torch_dtype=torch.float16
)

print("Qwen loaded successfully on GPU!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Qwen loaded successfully on GPU!


In [9]:
import time
import torch
from sentence_transformers import util

question = "What is the college's application fee?"

# Convert the question into an embedding
question_embedding = model.encode(
    question,
    convert_to_tensor=True
)

# Find the most relevant PDF chunks
similarities = util.cos_sim(
    question_embedding,
    chunk_embeddings
)[0]

top_k = min(3, len(chunks))
results = similarities.topk(top_k)

context = "\n\n".join(
    chunks[int(i)] for i in results.indices
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a PDF question-answering assistant. "
            "Answer using only the supplied document context. "
            "If the requested information is absent, say: "
            "'The document does not provide this information.' "
            "Do not guess or invent information. "
            "Keep answers short and accurate."
        )
    },
    {
        "role": "user",
        "content": (
            f"Document context:\n{context}\n\n"
            f"Question: {question}"
        )
    }
]

start = time.time()

response = llm(
    messages,
    max_new_tokens=60,
    do_sample=False
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("Generation time:", round(time.time() - start, 2), "seconds")
print("Answer:", response[0]["generated_text"][-1]["content"])

Generation time: 0.9 seconds
Answer: Rs. 1,15,000/-


In [10]:

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sentence_transformers import util

# Chatbot title
display(HTML("""
<div style="padding:18px; background:#17365D; color:white;
            border-radius:10px; margin-bottom:12px;">
    <h2 style="margin:0;">PDF Assistant</h2>
    <p style="margin:6px 0 0;">Ask questions about your college brochure</p>
</div>
"""))

# Question input
question_box = widgets.Text(
    placeholder="Example: What is the MBA fee?",
    description="Question:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%")
)

# Ask button
ask_button = widgets.Button(
    description="Ask Question",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="160px", height="40px")
)

# Answer area
output_box = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        border="1px solid #cccccc",
        padding="12px"
    )
)

# Prepare embeddings once
chunk_embeddings = model.encode(
    chunks,
    convert_to_tensor=True
)

def answer_question(_):
    with output_box:
        clear_output()

        question = question_box.value.strip()

        if not question:
            print("Please enter a question.")
            return

        question_embedding = model.encode(
            question,
            convert_to_tensor=True
        )

        similarities = util.cos_sim(
            question_embedding,
            chunk_embeddings
        )[0]

        top_k = min(3, len(chunks))
        results = similarities.topk(top_k)

        context = "\n\n".join(
            chunks[int(i)] for i in results.indices
        )

        messages = [
            {
                "role": "system",
                "content": (
                     "You are a PDF question-answering assistant. "
    "Answer using only the supplied document context. "
    "Find the exact information requested. "
    "If the information is absent from the context, say: "
    "'The document does not provide this information.' "
    "Do not guess or invent information. "
    "Keep answers short and accurate."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Document context:\n{context}\n\n"
                    f"Question: {question}"
                )
            }
        ]

        response = llm(
            messages,
            max_new_tokens=100,
            do_sample=False
        )

        answer = response[0]["generated_text"][-1]["content"]

        display(HTML("""
        <div style="background:#EAF4EA; padding:12px;
                    border-radius:8px; margin-bottom:8px;">
            <b>Answer</b>
        </div>
        """))
        print(answer)

ask_button.on_click(answer_question)

display(
    question_box,
    ask_button,
    output_box
)


Text(value='', description='Question:', layout=Layout(width='100%'), placeholder='Example: What is the MBA fee…

Button(button_style='primary', description='Ask Question', icon='search', layout=Layout(height='40px', width='…

Output(layout=Layout(border='1px solid #cccccc', padding='12px', width='100%'))

In [13]:
print("Model available:", "model" in globals())
print("LLM available:", "llm" in globals())
print("PDF chunks available:", "chunks" in globals())

Model available: True
LLM available: True
PDF chunks available: True


In [14]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

PyTorch version: 2.14.1+cu130
CUDA available: True
CUDA version: 13.0


In [15]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [16]:
print("Qwen loaded:", "llm" in globals())
print("PDF chunks available:", "chunks" in globals())
print("Embeddings available:", "chunk_embeddings" in globals())

Qwen loaded: True
PDF chunks available: True
Embeddings available: True


In [17]:
print("PDF text available:", "full_text" in globals())
print("PDF chunks available:", "chunks" in globals())

if "chunks" in globals():
    print("Number of chunks:", len(chunks))

PDF text available: True
PDF chunks available: True
Number of chunks: 5
